<a href="https://colab.research.google.com/github/aldo02032004/social-media-kol-analysis/blob/main/social_media_kol_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Social media KOL analysis

From one media-monitoring export to: the top issues in news vs social media, and the key opinion leaders (KOLs) of a topic with their stance.

1. **Clean**: separate news outlets from organic accounts, merge spellings of the same account, flag copypasta.
2. **Top issues**: the 5 news and 5 social posts with the most followers become 5 news issues and 5 social issues.
3. **Classify** each post with a topic keyword: relevant or not, theme, sentiment, stance toward the target (posts without a keyword skip the LLM).
4. **Network**: retweet / reply / mention graph per theme, scored by PageRank, eigenvector, engagement and followers.
5. **Summaries**: stance and narrative per KOL, then per theme.

The LLM is **Gemma 2 9B SahabatAI** (GGUF, ~5.8 GB) running on the Colab GPU: select **Runtime → Change runtime type → T4 GPU**. No API key. Config, cache and labels are saved to Google Drive.

## 1. Install

In [ ]:
# wheel llama-cpp CUDA 12 + library CUDA 12-nya (Colab sudah CUDA 13)
%pip install -q "great[text] @ git+https://github.com/azmkto/GREAT-Tools.git@6e126253da57688d951d1bb1d55441993c8842f3" emoji llama-cpp-python==0.3.36 nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## 2. Form

In [ ]:
PROJECT = "prabowo_vladivostok"   # nama file di Drive
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"
STANCE_TARGET = "Presiden Prabowo dan pemerintah"
TARGET_ACCOUNTS = "prabowo, prabowosubianto, gerindra"   # akun target: tetap node, tidak ikut ranking
SOURCES_TEXT = "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"
REGENERATE_TOPIC_CONFIG = False

INTERACTION_WEIGHT = {"retweet": 1.0, "reply": 0.5, "mention": 0.25}
SCORE_COLS = ["pagerank", "eigenvector", "total_engagement", "followers"]   # skor = rata-rata persentil
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15
COPYPASTA_MIN_ACCOUNTS = 3     # teks sama dari >= n akun = indikasi koordinasi
N_ISSUES = 5                   # isu utama per kategori (berita / media sosial)

import ctypes, json, re, unicodedata
from datetime import date
from functools import lru_cache
from pathlib import Path
from urllib.error import HTTPError

import emoji
import networkx as nx
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display
from scipy.sparse.linalg import eigsh
from tqdm.auto import tqdm

from great import EXPORT_COLUMNS, clean_for_bert, is_media_account
from great.labels import is_media_name

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
TOPIC, STANCE_TARGET = TOPIC.strip(), STANCE_TARGET.strip()
SOURCES = [s for s in re.split(r"[,\s]+", SOURCES_TEXT) if s]
assert PROJECT and TOPIC and STANCE_TARGET and SOURCES, "Form belum lengkap"

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 3. Load Gemma

In [ ]:
LLM_MODEL_REPO = "faizonly5953/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF"
LLM_MODEL_FILE = "gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf"
LLM_N_CTX = 8192

if not Path("/dev/nvidia0").exists():
    raise RuntimeError("Butuh GPU: Runtime -> Change runtime type -> T4 GPU, lalu jalankan ulang dari atas.")
import nvidia.cublas, nvidia.cuda_runtime   # wheel llama-cpp butuh CUDA 12: muat dulu sebelum import llama_cpp
for pkg, lib in [(nvidia.cuda_runtime, "libcudart.so.12"), (nvidia.cublas, "libcublasLt.so.12"), (nvidia.cublas, "libcublas.so.12")]:
    ctypes.CDLL(f"{pkg.__path__[0]}/lib/{lib}", mode=ctypes.RTLD_GLOBAL)
import llama_cpp

if "llm" not in globals():   # dimuat sekali; dimuat ulang = memori GPU habis
    llm = llama_cpp.Llama.from_pretrained(repo_id=LLM_MODEL_REPO, filename=LLM_MODEL_FILE,   # ~5,8 GB
                                          n_gpu_layers=-1, n_ctx=LLM_N_CTX, n_batch=512, verbose=False)


def ask_llm(content, schema, max_tokens=700):
    r = llm.create_chat_completion(messages=[{"role": "user", "content": content}],
                                   response_format={"type": "json_object", "schema": schema},
                                   temperature=0, max_tokens=max_tokens)
    return json.loads(r["choices"][0]["message"]["content"])


def fmt_posts(posts, max_words=120):
    return "\n".join(f"{i}. " + " ".join(str(p).split()[:max_words]) for i, p in enumerate(posts, 1))

## 4. Topic config (Gemma)

In [ ]:
TOPIC_CONFIG_PATH = Path(f"{DRIVE_DIR}/topic_config_{PROJECT}.json")
CONFIG_SCHEMA = {
    "type": "object",
    "properties": {
        "themes": {"type": "array", "minItems": 2, "maxItems": 4,
                   "items": {"type": "object", "properties": {"nama": {"type": "string"}, "deskripsi": {"type": "string"}},
                             "required": ["nama", "deskripsi"]}},
        "keywords": {"type": "array", "minItems": 20, "maxItems": 60, "items": {"type": "string"}},
        "media_accounts": {"type": "array", "maxItems": 30, "items": {"type": "string"}},
    },
    "required": ["themes", "keywords", "media_accounts"],
}
TOPIC_CONFIG_PROMPT = f"""Kamu analis media sosial Indonesia yang menyiapkan konfigurasi klasifikasi otomatis.
Topik: "{TOPIC}". Tanggal hari ini: {date.today()}. Jangan menulis tahun di keyword kecuali yakin.

Cara pakai: cuitan tanpa satu pun keyword dibuang tanpa dibaca AI (keyword terlewat = cuitan relevan hilang,
terlalu umum = ribuan cuitan tidak relevan ikut dibaca). Cuitan yang lolos diberi satu tema. Akun di media_accounts dibuang.
Data mentah mencakup SEMUA cuitan yang menyebut tokoh utama, termasuk isu lain, jadi nama tokoh utama BUKAN penanda topik.

themes: 2-4 tema per sudut pandang percakapan (agenda/acara, kebijakan/kerja sama, reaksi politik), bukan per tokoh.
- nama: snake_case bahasa Indonesia, bukan "lainnya"/"umum"/"campuran".
- deskripsi: "Masuk: <1 kalimat>. Contoh frasa: <8-15 frasa khas, termasuk gaul/singkatan/julukan>. Bukan: <yang sering tertukar>."

keywords: 20-60 kata/frasa pendek. Kata >=5 huruf juga cocok sebagai bagian kata lain (hashtag gabungan).
- Isi: penanda khas topik ini (pihak lain, tempat, acara, instansi; ejaan Indonesia & Inggris, singkatan, typo umum).
- DILARANG: nama/julukan tokoh utama sendirian, dan kata umum lintas isu ("investasi", "pertahanan", "kesepakatan",
  "geopolitik", "infrastruktur", "energi", "komoditas", "digitalisasi", "kunjungan kerja", "kebijakan luar negeri").

media_accounts: username media/kanal resmi institusi terkait, tanpa @. Bukan akun pribadi, buzzer, influencer. Ragu = jangan.

Output HANYA JSON."""

topic_config = json.loads(TOPIC_CONFIG_PATH.read_text(encoding="utf-8")) if TOPIC_CONFIG_PATH.exists() else {}
if REGENERATE_TOPIC_CONFIG or topic_config.get("_topic") != TOPIC:
    out = ask_llm(TOPIC_CONFIG_PROMPT, CONFIG_SCHEMA, max_tokens=2048)
    themes = {re.sub(r"\W+", "_", t["nama"].strip().lower()).strip("_"): t["deskripsi"].strip() for t in out["themes"]}
    themes.pop("lainnya", None)
    if not themes:
        raise RuntimeError("Gemma gagal bikin tema, jalankan ulang")
    topic_config = {"_topic": TOPIC, "themes": themes, "keywords": out["keywords"], "media_accounts": out["media_accounts"]}
    TOPIC_CONFIG_PATH.write_text(json.dumps(topic_config, ensure_ascii=False, indent=2), encoding="utf-8")

THEME_DESCRIPTIONS = {**topic_config["themes"], "lainnya": "topik di luar kategori di atas"}
THEMES = list(THEME_DESCRIPTIONS)
RELEVANT_THEMES = THEMES[:-1]
KW_RELEVAN = sorted({k.strip().lower() for k in topic_config["keywords"] if k.strip()}, key=len, reverse=True)
EXTRA_MEDIA_ACCOUNTS = {a.lower().lstrip("@") for a in topic_config["media_accounts"]}
# keyword >= 5 huruf juga cocok di dalam kata lain (hashtag gabungan)
RE_RELEVAN = re.compile("|".join(re.escape(k) if len(k) >= 5 else rf"(?<!\w){re.escape(k)}(?!\w)" for k in KW_RELEVAN))
print(json.dumps(topic_config, ensure_ascii=False, indent=2))

## 5. Load data

In [ ]:
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]


def xlsx_url(link):
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_raw_data(link):
    for attempt in range(3):   # Google kadang membalas 400 sesaat
        try:
            df = pd.read_excel(xlsx_url(link), header=1)
            break
        except HTTPError:
            if attempt == 2:
                raise
        except ValueError as e:
            raise ValueError(f"Gagal baca {link}, cek akses 'Anyone with the link'") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]   # KeyError = kolom kurang


df_all = pd.concat([load_raw_data(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)], ignore_index=True)
print(f"Data: {len(df_all)} baris, {len(SOURCES)} file")

## 6. Separate news outlets and clean text

In [ ]:
ALIASES = {"kompastvcom": "kompascom", "hippoacademyid": "hippoacademy"}   # gabung manual
TWO_PART_SLD = {"co", "or", "ac", "go", "web", "my", "sch", "net", "com"}


@lru_cache(maxsize=None)   # habis edit ALIASES, jalankan ulang
def canonical(name):
    """Beda ejaan akun (@kompascom, kompas.com) -> satu key."""
    if pd.isna(name):
        return None
    s = re.sub(r"^(https?://)?(www\.)?@?", "", unicodedata.normalize("NFKC", str(name)).strip().lower()).split("/")[0]
    parts = s.split(".")
    if len(parts) > 2 and re.fullmatch(r"[a-z]{2,3}", parts[-1]):   # domain, bukan handle IG bertitik
        parts = parts[-3:] if parts[-2] in TWO_PART_SLD and len(parts[-1]) == 2 else parts[-2:]
    key = re.sub(r"[^a-z0-9]", "", "".join(parts)) or re.sub(r"\W", "", s)   # non-latin tetap hurufnya
    return ALIASES.get(key, key) if key else None


TARGET_IDS = {canonical(a) for a in TARGET_ACCOUNTS.split(",")} - {None}

NUM_COLS = ["Followers", "Retweeted", "Favourited"]
df_all[NUM_COLS] = df_all[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
df_all["author_key"] = df_all["Author"].map(canonical)

# media = platform berita, akun media, atau ejaan lain dari akun media
is_media = pd.Series([is_media_account(a, p, extra_accounts=EXTRA_MEDIA_ACCOUNTS)
                      for a, p in zip(df_all["Author"], df_all["Media"])], index=df_all.index)
is_media |= df_all["Author"].map(is_media_name)
MEDIA_IDS = set(df_all.loc[is_media, "author_key"].dropna()) | {canonical(a) for a in EXTRA_MEDIA_ACCOUNTS}
is_media |= df_all["author_key"].isin(MEDIA_IDS)
df_all["is_media"] = is_media

# RT ditulis "RT <teks>" (sering tanpa @user), reply "... [RE username]"
RE_RT_PREFIX = re.compile(r"^\s*RT\b\s*(?:@\w+\s*:\s*)?")
RE_RT = re.compile(r"(?<!\w)RT\s*@(\w+)", re.IGNORECASE)
RE_REPLY = re.compile(r"\[RE\s+@?(\w+)\]", re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")


def strip_noise(text):
    text = RE_REPLY.sub(" ", RE_RT_PREFIX.sub("", text))
    text = emoji.demojize(text, language="id").replace("_", " ").replace(":", " ")
    return RE_ELONGATION.sub(r"\1\1", text)


def build_raw_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


def dedup_key(texts):
    return texts.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")


def match_key(texts, n_words=12):
    """12 kata pertama: RT yang terpotong ("...") tetap ketemu aslinya."""
    return dedup_key(texts).str.split().str[:n_words].str.join(" ")


df_all["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(df_all["Headline"], df_all["Mentions"])]
df_all["is_retweet"] = df_all["text_raw_combined"].map(lambda t: bool(RE_RT_PREFIX.match(t)))
df_all["text_clean"] = [clean_for_bert(strip_noise(t)) for t in tqdm(df_all["text_raw_combined"], desc="Cleaning")]
df_raw = df_all[~df_all["is_media"]].reset_index(drop=True)   # akun organik: bahan jaringan & KOL
print(f"Media: {df_all['is_media'].sum()}, organik: {len(df_raw)}")

# copypasta: teks non-RT sama persis dari banyak akun. ditandai, tetap dihitung
key = dedup_key(df_raw["text_clean"])
own = ~df_raw["is_retweet"] & key.str.split().str.len().ge(3)
n_accounts = df_raw[own].groupby(key[own])["author_key"].nunique()
df_raw["copypasta"] = own & key.map(n_accounts).ge(COPYPASTA_MIN_ACCOUNTS)
print(f"Copypasta: {df_raw['copypasta'].sum()} baris dari {n_accounts.ge(COPYPASTA_MIN_ACCOUNTS).sum()} teks")


def unique_posts(df):
    """Teks unik (>= 3 kata), post asli didahulukan dari RT. Dipakai untuk bacaan LLM."""
    return (df[df["text_clean"].str.split().str.len() >= 3]
            .assign(dedup_key=lambda d: dedup_key(d["text_clean"]))
            .sort_values("is_retweet", kind="stable")
            .drop_duplicates(subset="dedup_key").sort_index()
            .drop(columns="dedup_key").reset_index(drop=True))


df_clean = unique_posts(df_raw)
print(f"Unik: {len(df_clean)} dari {len(df_raw)}")

## 7. Top issues: news vs social media

In [ ]:
PERSONA = """You're Syahganda Nainggolan — seorang intelektual-aktivis Indonesia, mantan aktivis ITB era 80-an
yang di-DO karena menentang Orde Baru, meraih gelar Doktor di bidang perburuhan dari FISIP UI,
dan dikenal sebagai tokoh oposisi yang konsisten mengkritik kekuasaan yang sedang berkuasa.
Kamu adalah salah satu petinggi KAMI (Koalisi Aksi Menyelamatkan Indonesia), pendiri
Sabang-Merauke Circle (SMC), dan punya concern besar pada isu buruh, kesejahteraan sosial,
dan nasionalisme.

Cara berpikirmu:
- Kritis dan skeptis terhadap narasi resmi pemerintah/elite penguasa.
- Tajam membedah ketimpangan sosial, nasib buruh/rakyat kecil, dan ancaman terhadap nasionalisme.
- Lugas, berani, dan konfrontatif.
- Mencari pola dan motif di balik data, bukan sekadar mendeskripsikan permukaan teks."""
ISSUE = {"type": "object",
         "properties": {"isu_utama": {"type": "string", "maxLength": 120},
                        "ringkasan": {"type": "string", "maxLength": 600},
                        "sumber": {"type": "array", "maxItems": 2 * N_ISSUES, "items": {"type": "integer"}},
                        "tokoh": {"type": "string", "maxLength": 200}},
         "required": ["isu_utama", "ringkasan", "sumber", "tokoh"]}
ISSUES_SCHEMA = {"type": "object",
                 "properties": {k: {"type": "array", "minItems": N_ISSUES, "maxItems": N_ISSUES, "items": ISSUE}
                                for k in ("berita_online", "media_sosial")},
                 "required": ["berita_online", "media_sosial"]}

# post unik dengan followers terbanyak per kategori, 1 post per akun
picked = pd.concat([unique_posts(df_all[df_all["is_media"] == news]).sort_values("Followers", ascending=False)
                    .drop_duplicates("author_key").head(N_ISSUES)
                    .assign(kategori="Berita/Online" if news else "Media Sosial") for news in (True, False)],
                   ignore_index=True)
picked.index += 1   # nomor item di prompt
items = "\n\n".join(f"[{i} | {r.kategori} | {r.Media} | {r.Author}]\n" + " ".join(r.text_raw_combined.split()[:120])
                     for i, r in picked.iterrows())
issues = ask_llm(f"""{PERSONA}

Berikut {N_ISSUES} item Berita/Online dan {N_ISSUES} item Media Sosial, masing-masing bernomor.
Buat {N_ISSUES} isu utama berita online dan {N_ISSUES} isu utama media sosial yang dianalisis secara kritis.
Kalau isu yang benar-benar berbeda kurang dari {N_ISSUES}, pecah sudut pandang/sub-isu dari data yang sama.
- ringkasan: maksimal 3 kalimat.
- sumber: nomor item yang dibahas.
- tokoh: tokoh publik/politik yang paling disorot, pisahkan koma; "Tidak ada tokoh spesifik" kalau tidak ada.

{items}

Output HANYA JSON.""", ISSUES_SCHEMA, max_tokens=3000)


def issue_sentiment(rows):
    """Dari kolom Sentiment export: skor = porsi positif; selisih <= 20% = kontroversial."""
    s = rows["Sentiment"].astype(str).str.strip().str.lower()
    pos, neg = s.isin(["positif", "positive", "pos"]).sum(), s.isin(["negatif", "negative", "neg"]).sum()
    if pos + neg == 0:
        return "Tidak diketahui"
    if abs(pos - neg) / (pos + neg) <= 0.2:
        return f"{pos / (pos + neg):.2f} (Kontroversial)"
    return f"{pos / (pos + neg):.2f} ({'Positif' if pos > neg else 'Negatif'})"


def issue_kol(rows):
    social = rows[rows["kategori"] == "Media Sosial"]
    if social.empty:
        return "Tidak ada (isu didominasi sumber berita)"
    top = social.loc[social["Followers"].idxmax()]
    return f"{top['Author']} ({top['Followers']:,.0f} followers)".replace(",", ".")


for kind, title in (("berita_online", "Isu berita online"), ("media_sosial", "Isu media sosial")):
    print(f"\n=== {title} ===")
    display(pd.DataFrame([{"isu": x["isu_utama"],
                           "sentimen": issue_sentiment(picked.loc[picked.index.intersection(x["sumber"])]),
                           "key_opinion_leader": issue_kol(picked.loc[picked.index.intersection(x["sumber"])]),
                           "ringkasan": x["ringkasan"], "tokoh": x["tokoh"]} for x in issues[kind]],
                         index=range(1, N_ISSUES + 1)))

## 8. Classify each post

In [ ]:
SENTIMENTS = ("positif", "negatif", "netral", "kontroversi")
STANCES = ("supporter", "moderat", "oposisi", "tidak_jelas")
UNCLEAR = "tidak_jelas"
THEME_CACHE_PATH = Path(f"{DRIVE_DIR}/theme_cache_{PROJECT}.json")
POST_SCHEMA = {"type": "object",
               "properties": {"membahas_topik": {"type": "boolean"},   # field pertama: diputuskan sebelum tema
                              "primary_theme": {"type": "string", "enum": RELEVANT_THEMES},
                              "sentiment": {"type": "string", "enum": list(SENTIMENTS)},
                              "stance": {"type": "string", "enum": list(STANCES)}},
               "required": ["membahas_topik", "primary_theme", "sentiment", "stance"]}
THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in RELEVANT_THEMES)

# prefix tetap -> llama-cpp tidak membaca ulang prompt untuk tiap cuitan
CLASSIFY_PREFIX = f"""Kamu classifier cuitan bahasa Indonesia tentang: {TOPIC}.
Pihak yang dinilai: {STANCE_TARGET}

Daftar tema:
{THEME_LIST_STR}

Untuk cuitan di bawah:
1. membahas_topik: true HANYA kalau isi utamanya tentang peristiwa di topik itu atau dampak langsungnya.
   false kalau membahas isu lain walau menyebut tokoh, negara, acara, atau istilah yang sama.
2. primary_theme: tema paling cocok.
3. sentiment: SIKAP penulis ke topik, BUKAN nada bahasanya.
   positif = mendukung/memuji/membela; negatif = mengkritik/menolak/menyindir;
   netral = hanya informasi; kontroversi = dukungan & kritik sama kuat.
   Membela {STANCE_TARGET} sambil memaki pengkritiknya tetap positif.
4. stance: sikap penulis ke {STANCE_TARGET}. Penentunya siapa yang dibela/diserang, BUKAN nada kalimat.
   supporter = mendukung/memuji/membela, termasuk menyerang pengkritik {STANCE_TARGET};
   oposisi = mengkritik/menolak/menyerang {STANCE_TARGET} atau kebijakannya;
   moderat = menimbang dua sisi, mengkritik kedua kubu, atau kritik tanpa menolak;
   tidak_jelas = hanya informasi/berita/dampak, tanpa memuji atau menyalahkan {STANCE_TARGET}.
   Kalimat kasar/negatif belum tentu oposisi; cek dulu siapa sasarannya.

"""
df_clean[["theme", "sentiment", "stance"]] = None

# 1) tanpa keyword -> 'lainnya'. keyword di >20% cuitan diabaikan (terlalu umum)
found = df_clean["text_clean"].str.lower().str.findall(RE_RELEVAN).map(set)
counts = found.explode().value_counts()
too_common = set(counts[counts > 0.2 * len(df_clean)].index)
if too_common:
    print(f"Keyword diabaikan (terlalu umum): {sorted(too_common)}")
review = found.map(lambda s: bool(s - too_common))
df_clean.loc[~review, "theme"] = "lainnya"
texts = df_clean.loc[review, "text_clean"]
if len(texts) > 0.5 * len(df_clean):
    raise RuntimeError(f"{len(texts)}/{len(df_clean)} lolos prefilter, keyword terlalu umum")

# 2) cache, valid kalau prompt, schema, & model sama
cache_id = [CLASSIFY_PREFIX, POST_SCHEMA, LLM_MODEL_FILE]
cached = json.loads(THEME_CACHE_PATH.read_text(encoding="utf-8")) if THEME_CACHE_PATH.exists() else {}
theme_cache = cached["texts"] if cached.get("id") == cache_id else {}
todo = [t for t in texts.unique() if t not in theme_cache]
print(f"Prefilter: {len(df_clean) - len(texts)} lainnya, {len(texts)} ke Gemma ({len(todo)} baru)")

# 3) 1 request per teks unik; cache disimpan walau terputus
try:
    for text in tqdm(todo, desc="Gemma"):
        content = CLASSIFY_PREFIX + f'Cuitan: "{" ".join(str(text).split()[:120])}"\n\nOutput HANYA JSON.'
        try:
            out = ask_llm(content, POST_SCHEMA, max_tokens=80)
            theme_cache[text] = [out["membahas_topik"], out["primary_theme"], out["sentiment"], out["stance"]]
        except (ValueError, KeyError) as e:   # jawaban rusak dilewati
            print(f"Skip {text[:50]!r}: {e}")
finally:
    THEME_CACHE_PATH.write_text(json.dumps({"id": cache_id, "texts": theme_cache}, ensure_ascii=False), encoding="utf-8")

# 4) label ke df_clean; tidak membahas topik -> 'lainnya'
hits = texts.map(theme_cache).dropna()
labels = pd.DataFrame(hits.tolist(), index=hits.index, columns=["relevan", "theme", "sentiment", "stance"])
df_clean.loc[labels.index, ["sentiment", "stance"]] = labels[["sentiment", "stance"]]
df_clean.loc[labels.index, "theme"] = labels["theme"].where(labels["relevan"], "lainnya")
if df_clean["theme"].isna().any():
    print(f"{df_clean['theme'].isna().sum()} baris belum ada tema, jalankan ulang")
print(df_clean["theme"].value_counts(dropna=False).to_string())

## 9. Network and KOL score

In [ ]:
HANDLE = re.compile(r"(?<![\w.])@([\w.]*\w)")   # handle IG bertitik ikut
EXCLUDE_IDS = TARGET_IDS | {"metaai", "youtube", "instagram", "tiktok", "facebook", "twitter", "threads"}


def extract_targets(text, retweet_of=None):
    """1 edge per akun tujuan, prioritas retweet > reply > mention."""
    rt = RE_RT.search(text)
    source = canonical(rt.group(1)) if rt else retweet_of
    targets = {source: "retweet"} if source else {}
    for m in RE_REPLY.findall(text):
        targets.setdefault(canonical(m), "reply")
    for m in HANDLE.findall(text):
        targets.setdefault(canonical(m), "mention")
    return list(targets.items())


def undirected(pair_w):
    """A->B + B->A dijumlah."""
    pair = np.sort(pair_w[["Source", "Target"]].to_numpy(), axis=1)
    und = pair_w.assign(u=pair[:, 0], v=pair[:, 1]).groupby(["u", "v"], as_index=False)["Weight"].sum()
    return nx.from_pandas_edgelist(und, "u", "v", edge_attr="Weight")


def eigenvector_all(U):
    """Per klaster, diskala (eigen klaster / eigen terbesar) supaya klaster kecil tetap dapat nilai."""
    parts = []
    for comp in nx.connected_components(U):
        nodes = list(comp)
        A = nx.to_scipy_sparse_array(U, nodelist=nodes, weight="Weight", dtype=float)
        val, vec = eigsh(A, k=1, which="LA", v0=np.ones(len(nodes)))
        parts.append((val[0], pd.Series(np.abs(vec[:, 0]), index=nodes)))
    lam_max = max(lam for lam, _ in parts)
    return pd.concat([vec * lam / lam_max for lam, vec in parts])


def centrality(pair_w):
    G = nx.from_pandas_edgelist(pair_w, "Source", "Target", edge_attr="Weight", create_using=nx.DiGraph)
    out = pd.DataFrame({"pagerank": pd.Series(nx.pagerank(G, weight="Weight"))})
    out["eigenvector"] = eigenvector_all(undirected(pair_w)).reindex(out.index, fill_value=0.0)
    return out.rename_axis("Id").reset_index()


def is_excluded(key):
    """Target, platform, media: tetap node, bukan KOL."""
    return key in EXCLUDE_IDS or key in MEDIA_IDS or is_media_account(key, extra_accounts=EXTRA_MEDIA_ACCOUNTS)


def top_stance(s):
    """Modus stance; 'tidak_jelas' kalau tidak ada sikap sama sekali."""
    known = s.dropna()[lambda x: x != UNCLEAR]
    return known.mode().iloc[0] if len(known) else UNCLEAR


df_topic = df_clean[df_clean["theme"].isin(RELEVANT_THEMES)]
if df_topic.empty:
    raise RuntimeError("Tidak ada cuitan relevan")

# label ke semua baris df_raw: RT ikut label teks yang di-RT
raw_keys = match_key(df_raw["text_clean"])
by_key = df_topic.assign(k=match_key(df_topic["text_clean"])).drop_duplicates("k").set_index("k")
df_raw[["theme", "sentiment", "stance"]] = by_key[["theme", "sentiment", "stance"]].reindex(raw_keys).to_numpy()

# RT tanpa @user: penulis asli = akun pertama yang memposting teks itu
is_rt = df_raw["is_retweet"]
orig_author = (df_raw.assign(k=raw_keys)[~is_rt].dropna(subset=["author_key"])
               .drop_duplicates("k").set_index("k")["author_key"])
retweet_of = raw_keys.map(orig_author).where(is_rt)

edge_rows = pd.DataFrame([
    {"row": i, "Source": author, "Target": target, "interaction": kind, "theme": theme}
    for i, author, raw, theme, rt_of in zip(df_raw.index, df_raw["author_key"], df_raw["text_raw_combined"],
                                            df_raw["theme"], retweet_of)
    if pd.notna(theme) and pd.notna(author)
    for target, kind in extract_targets(raw, rt_of if pd.notna(rt_of) else None)
    if target != author
], columns=["row", "Source", "Target", "interaction", "theme"])
df_edges = edge_rows.groupby(["Source", "Target", "interaction", "theme"]).size().rename("Weight").reset_index()
print(f"RT ketemu aslinya: {retweet_of.notna().sum()}/{is_rt.sum()}")
print("Edges:", df_edges.groupby("interaction")["Weight"].sum().to_dict())

df_raw["engagement"] = df_raw["Favourited"] + df_raw["Retweeted"]
posted = df_raw[df_raw["theme"].notna() & df_raw["author_key"].notna()]

# volume dari post asli (Retweeted di baris RT milik tweet asli); dari df_raw supaya akun copypasta tetap terhitung
agg = (posted[~posted["is_retweet"]].groupby(["theme", "author_key"])
       .agg(jumlah_post=("engagement", "size"), total_engagement=("engagement", "sum"), copypasta=("copypasta", "mean"))
       .reset_index().rename(columns={"author_key": "Id"}))

# bobot per jenis interaksi, log supaya 1 akun RT 50x != 50 akun
w_edges = (df_edges.assign(Weight=df_edges["Weight"] * df_edges["interaction"].map(INTERACTION_WEIGHT))
           .groupby(["theme", "Source", "Target"], as_index=False)["Weight"].sum())
w_edges["Weight"] = np.log1p(w_edges["Weight"])
net_parts = [centrality(sub).assign(theme=theme) for theme, sub in w_edges.groupby("theme")]
df_net = pd.concat(net_parts, ignore_index=True) if net_parts else pd.DataFrame(columns=["Id", "pagerank", "eigenvector", "theme"])

display_name = (df_raw.value_counts(["author_key", "Author"]).reset_index()
                .drop_duplicates("author_key").set_index("author_key")["Author"])
followers_by_id = df_raw.groupby("author_key")["Followers"].max()

kol = agg.merge(df_net, on=["theme", "Id"], how="outer")
kol = kol[~kol["Id"].map(is_excluded).astype(bool)].copy()
retweeted = edge_rows[edge_rows["interaction"] == "retweet"]
has_text = set(zip(posted["theme"], posted["author_key"])) | set(zip(retweeted["theme"], retweeted["Target"]))
kol = kol[[p in has_text for p in zip(kol["theme"], kol["Id"])]].copy()
num = ["jumlah_post", "total_engagement", "copypasta", "pagerank", "eigenvector"]
kol[num] = kol[num].fillna(0)
kol["followers"] = kol["Id"].map(followers_by_id).fillna(0)
kol["author_id"] = kol["Id"].map(display_name).fillna(kol["Id"])
kol["score"] = kol.groupby("theme")[SCORE_COLS].rank(pct=True).mean(axis=1)   # persentil: tahan heavy tail

df_top_authors = (kol.sort_values(["theme", "score"], ascending=[True, False])
                  .groupby("theme").head(TOP_N_AUTHORS_PER_THEME).reset_index(drop=True))
df_top_authors["rank"] = df_top_authors.groupby("theme").cumcount() + 1

node_stance = posted.groupby(["theme", "author_key"])["stance"].agg(top_stance)
top_pairs = set(zip(df_top_authors["theme"], df_top_authors["Id"]))
pairs = list(zip(df_net["theme"], df_net["Id"]))
df_nodes = df_net.assign(akun=df_net["Id"].map(display_name).fillna(df_net["Id"]),
                         followers=df_net["Id"].map(followers_by_id),
                         stance=[node_stance.get(p) for p in pairs],
                         is_target=df_net["Id"].isin(TARGET_IDS),
                         is_kol=[p in top_pairs for p in pairs])
print(f"KOL {len(df_top_authors)}, nodes {len(df_nodes)}, edges {len(df_edges)}")

## 10. KOL and theme summaries (Gemma)

In [ ]:
SUMMARY_SCHEMA = {"type": "object", "properties": {"summary": {"type": "string", "maxLength": 600}}, "required": ["summary"]}
KOL_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}

Cuitan & retweet akun "{author}" di tema ini:
{posts}

summary: 2-3 kalimat pandangan/narasi utama akun ini soal tema tersebut, jangan menyalin cuitan.
Bahasa Indonesia. Output HANYA JSON."""

rows_by_kol = posted.groupby(["theme", "author_key"]).groups
rt_rows = retweeted.groupby(["theme", "Target"])["row"].agg(list).to_dict()

records = []
for r in tqdm(df_top_authors.itertuples(), total=len(df_top_authors), desc="KOL"):
    # cuitan & RT akun ini; kalau kosong, teks dia yang di-RT orang
    rows = df_raw.loc[list(rows_by_kol.get((r.theme, r.Id), [])) or rt_rows.get((r.theme, r.Id), [])]
    stance = top_stance(rows["stance"])
    posts = (rows.sort_values("engagement", ascending=False)["text_clean"]
             .drop_duplicates().head(MAX_POSTS_PER_AUTHOR_SUMMARY).tolist())
    summary = ""
    if posts:
        try:
            summary = ask_llm(KOL_PROMPT.format(topic=TOPIC, theme=r.theme.replace("_", " "), author=r.author_id,
                                                posts=fmt_posts(posts)), SUMMARY_SCHEMA)["summary"].strip()
        except (ValueError, KeyError) as e:
            print(f"{r.author_id} ({r.theme}) gagal: {e}")
    records.append({"theme": r.theme, "rank": r.rank, "author_id": r.author_id, "Id": r.Id, "score": r.score,
                    "jumlah_teks": len(rows),
                    "sentiment": rows["sentiment"].mode().iloc[0] if rows["sentiment"].notna().any() else UNCLEAR,
                    "stance": stance,
                    "stance_share": rows["stance"].eq(stance).mean() if len(rows) else 0.0,   # confidence
                    "copypasta": r.copypasta, "summary": summary})
df_kol = pd.DataFrame(records)

THEME_FIELDS = ["ringkasan", "narasi_supporter", "narasi_moderat", "narasi_oposisi"]
THEME_SCHEMA = {"type": "object", "properties": {f: {"type": "string", "maxLength": 800} for f in THEME_FIELDS},
                "required": THEME_FIELDS}
THEME_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}
Komposisi sikap KOL: {composition}

Ringkasan KOL, urut dari paling berpengaruh:
{kols}

Tugas (HANYA dari ringkasan di atas, tanpa fakta luar):
1. ringkasan: 3-4 kalimat isi & arah percakapan di tema ini.
2. narasi_supporter: 1-2 kalimat argumen utama supporter, "-" kalau tidak ada.
3. narasi_moderat: 1-2 kalimat argumen utama moderat, "-" kalau tidak ada.
4. narasi_oposisi: 1-2 kalimat argumen utama oposisi, "-" kalau tidak ada.

Bahasa Indonesia. Output HANYA JSON."""

theme_records = []
for theme, kols in df_kol[df_kol["summary"] != ""].sort_values("rank").groupby("theme"):
    counts = kols["stance"].value_counts()
    content = THEME_PROMPT.format(
        topic=TOPIC, theme=theme.replace("_", " "), target=STANCE_TARGET,
        composition=", ".join(f"{s} {counts.get(s, 0)}" for s in STANCES),
        kols="\n".join(f"- [{k.stance}] {k.author_id}: {k.summary}" for k in kols.itertuples()))
    try:
        res = ask_llm(content, THEME_SCHEMA)
        out = {f: res[f].strip() for f in THEME_FIELDS}
    except (ValueError, KeyError) as e:
        print(f"Tema {theme} gagal: {e}")
        out = dict.fromkeys(THEME_FIELDS, "")
    theme_records.append({"theme": theme, "jumlah_kol": len(kols), **{s: int(counts.get(s, 0)) for s in STANCES}, **out})
df_theme_summary = pd.DataFrame(theme_records)

## 11. Results

In [ ]:
STANCE_LABEL = {"supporter": "Mendukung", "moderat": "Berimbang", "oposisi": "Menentang", "tidak_jelas": "Tanpa sikap"}
SENT_LABEL = {"positif": "Positif", "negatif": "Negatif", "netral": "Netral", "kontroversi": "Pro & kontra"}
nice = lambda s: s.str.replace("_", " ").str.capitalize()   # nama tema enak dibaca

n = df_kol["jumlah_teks"]
view = pd.DataFrame({
    "Tema": nice(df_kol["theme"]),
    "#": df_kol["rank"],
    "Akun": df_kol["author_id"],
    "Skor pengaruh": df_kol["score"].round(2),
    f"Sikap ke {STANCE_TARGET}": df_kol["stance"].map(STANCE_LABEL).where(n > 0, "Tidak ada cuitan"),
    "Dasar sikap": [f"{round(s * k)}/{k} cuitan" if k else "-" for s, k in zip(df_kol["stance_share"], n)],
    "Nada ke topik": df_kol["sentiment"].map(SENT_LABEL).fillna("-"),
    "Ringkasan narasi": df_kol["summary"],
})
if df_kol["copypasta"].gt(0).any():
    view.insert(6, "Copypasta", df_kol["copypasta"].map("{:.0%}".format))
print("Skor pengaruh: rata-rata persentil PageRank, eigenvector, engagement, followers di tema (1 = tertinggi)")
print("Dasar sikap: cuitan yang sikapnya sama / total cuitan yang dibaca")
display(view)

if len(df_theme_summary):
    print("\nRingkasan tema:")
    display(df_theme_summary.assign(theme=nice(df_theme_summary["theme"])).rename(columns={
        "theme": "Tema", "jumlah_kol": "KOL", **STANCE_LABEL, "ringkasan": "Ringkasan",
        "narasi_supporter": "Narasi pendukung", "narasi_moderat": "Narasi berimbang", "narasi_oposisi": "Narasi penentang"}))

print("\nTop 10 akun per tema di jaringan:")
nodes = df_nodes.sort_values(["theme", "pagerank"], ascending=[True, False]).groupby("theme").head(10)
display(pd.DataFrame({"Tema": nice(nodes["theme"]), "Akun": nodes["akun"],
                      "PageRank": nodes["pagerank"].round(3), "Eigenvector": nodes["eigenvector"].round(3),
                      "Followers": nodes["followers"].map(lambda f: "-" if pd.isna(f) else f"{f:,.0f}"),
                      "Sikap": nodes["stance"].map(STANCE_LABEL).fillna("-"),
                      "Peran": np.select([nodes["is_target"], nodes["is_kol"]], ["Target", "KOL"], "")}))